## Supplementary logit model: probability of rent decline

As an additional analysis, we estimate a logit model where the dependent variable equals 1 if the average rent price decreased relative to the previous month, and 0 otherwise.

The model is:

$$
\Pr(price\_drop_{itk}=1)=\Lambda\left(\alpha+\beta_1 total\_attacks_{it}+\beta_2 fatalities_{it}+\beta_3 unemp\_monthly_{it}+\mu_i+\lambda_t+\delta_k\right)
$$

where:

- $price\_drop_{itk}=1$ if current rent is lower than rent in the previous month,
- $\mu_i$ are region fixed effects,
- $\lambda_t$ are month fixed effects,
- $\delta_k$ are apartment-type fixed effects.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv("rent_acled_with_lags.csv")

df = df.rename(columns={
    "область": "region",
    "тип_квартири": "flat_type",
    "середня_ціна_грн": "rent_uah"
})

df["region"] = df["region"].astype(str)
df["month"] = df["month"].astype(str)
df["flat_type"] = df["flat_type"].astype(str)

df["price_drop"] = np.where(df["rent_uah"] < df["rent_lag1"], 1, 0)

logit_data = df[
    ["price_drop", "total_attacks", "fatalities", "unemp_monthly", "region", "month", "flat_type", "rent_lag1"]
].dropna().copy()

print(logit_data.shape)
print(logit_data["price_drop"].value_counts())
logit_data.head()

(2303, 8)
price_drop
0    1200
1    1103
Name: count, dtype: int64


,price_drop,total_attacks,fatalities,unemp_monthly,region,month,flat_type,rent_lag1
1,0,4,0,8456.0,Івано-Франківська,2022-03,1-кімнатна квартира,6620.0
2,1,0,0,8456.0,Івано-Франківська,2022-04,1-кімнатна квартира,6800.0
3,0,0,0,8456.0,Івано-Франківська,2022-05,1-кімнатна квартира,5725.0
4,1,0,0,8456.0,Івано-Франківська,2022-06,1-кімнатна квартира,8779.0
5,0,0,0,8456.0,Івано-Франківська,2022-07,1-кімнатна квартира,7831.0


In [2]:
# logit model with region, month, and apartment-type fixed effects
logit_model = smf.glm(
    formula="price_drop ~ total_attacks + fatalities + unemp_monthly + C(region) + C(month) + C(flat_type)",
    data=logit_data,
    family=sm.families.Binomial()
).fit(
    cov_type="cluster",
    cov_kwds={"groups": logit_data["region"]}
)

print(logit_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             price_drop   No. Observations:                 2303
Model:                            GLM   Df Residuals:                     2242
Model Family:                Binomial   Df Model:                           60
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1533.9
Date:                Tue, 14 Apr 2026   Deviance:                       3067.9
Time:                        22:16:07   Pearson chi2:                 2.31e+03
No. Iterations:                     4   Pseudo R-squ. (CS):            0.05106
Covariance Type:              cluster                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [3]:
# main coefficients only
coef_table = pd.DataFrame({
    "coef": logit_model.params,
    "std_err": logit_model.bse,
    "z_stat": logit_model.tvalues,
    "p_value": logit_model.pvalues,
    "odds_ratio": np.exp(logit_model.params)
})

coef_table.loc[["total_attacks", "fatalities", "unemp_monthly"]]

,coef,std_err,z_stat,p_value,odds_ratio
total_attacks,0.000058,0.000649,0.089632,0.928580,1.000058
fatalities,-0.000197,0.000229,-0.862783,0.388257,0.999803
unemp_monthly,0.000027,0.000018,1.521315,0.128181,1.000027


## Interpretation

- A positive coefficient means that the variable increases the probability of rent decline.
- A negative coefficient means that the variable decreases the probability of rent decline.
- If the p-value is below 0.05, the effect is statistically significant.
- The odds ratio shows how the odds of rent decline change when the explanatory variable increases by one unit.